In [1]:
import torch
import re
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModel
from natasha import Doc, NewsEmbedding, NewsNERTagger, NewsMorphTagger, Segmenter
import stanza
from collections import defaultdict
import pandas as pd
class TextVectorizer:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Инициализация компонентов Natasha
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)
        self.morph_tagger = NewsMorphTagger(self.emb)
        
        # Инициализация NER пайплайна
        self.ner_pipeline = pipeline(
            "ner",
            model="Gherman/bert-base-NER-Russian",
            tokenizer="Gherman/bert-base-NER-Russian",
            device=0 if torch.cuda.is_available() else -1,
            aggregation_strategy="simple"
        )
        
        # Инициализация Stanza
        self.nlp = stanza.Pipeline(
            lang='ru', 
            processors='tokenize,lemma,ner', 
            use_gpu=True,
            tokenize_pretokenized=True
        )
        
        # Загрузка модели SBERT
        model_name = 'ai-forever/sbert_large_mt_nlu_ru'
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
    
    def _neural_normalize(self, text):
        """Лемматизация текста с помощью Stanza"""
        doc = self.nlp(text)
        return " ".join([word.lemma for sent in doc.sentences for word in sent.words])
    
    def _replace_entities(self, text):
        """Замена именованных сущностей"""
        try:
            text = str(text).strip()
            if not text:
                return text
            
            entities = []
            
            # Извлечение сущностей из обеих моделей
            hf_entities = self.ner_pipeline(text)
            for ent in hf_entities:
                if isinstance(ent, dict):
                    entities.append((ent['start'], ent['end'], ent['entity_group']))
            
            doc = Doc(text)
            doc.segment(self.segmenter)
            doc.tag_ner(self.ner_tagger)
            for span in doc.spans:
                entities.append((span.start, span.stop, span.type))
            
            # Сортировка и замена
            entities = sorted(entities, key=lambda x: x[1]-x[0], reverse=True)
            counters = defaultdict(int)
            replacements = []
            
            for start, end, ent_type in entities:
                counters[ent_type] += 1
                replacement = f"{ent_type}-{chr(64 + counters[ent_type])}"
                replacements.append((start, end, replacement))
            
            text_list = list(text)
            for start, end, replacement in sorted(replacements, reverse=True, key=lambda x: x[0]):
                text_list[start:end] = list(replacement)
            
            return ''.join(text_list)
        
        except Exception as e:
            print(f"Ошибка замены сущностей: {str(e)}")
            return text
    
    def _get_embeddings(self, texts, batch_size=8):
        """Генерация эмбеддингов для списка текстов"""
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_embeddings)
            
            del inputs, outputs
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
        
        return np.array(embeddings)
    
    def process(self, text):
        """
        Основной метод обработки текста
        Возвращает усредненный вектор-эмбеддинг
        """
        # Шаг 1: Нормализация текста
        normalized = self._neural_normalize(text)
        
        # Шаг 2: Замена сущностей
        ner_replaced = self._replace_entities(text)
        
        # Шаг 3: Генерация эмбеддингов
        embeddings = self._get_embeddings([text, normalized, ner_replaced], batch_size=3)
        
        # Шаг 4: Усреднение эмбеддингов
        return np.mean(embeddings, axis=0)

C:\Users\maksi\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vectorizer = TextVectorizer()


Device set to use cuda:0
2025-05-19 23:57:57 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-05-19 23:57:58 INFO: Downloaded file to C:\Users\maksi\stanza_resources\resources.json
2025-05-19 23:57:58 INFO: Loading these models for language: ru (Russian):
| Processor | Package            |
----------------------------------
| tokenize  | syntagrus          |
| lemma     | syntagrus_nocharlm |
| ner       | wikiner            |

2025-05-19 23:57:58 INFO: Using device: cuda
2025-05-19 23:57:58 INFO: Loading: tokenize
2025-05-19 23:57:58 INFO: Loading: lemma
2025-05-19 23:58:00 INFO: Loading: ner
2025-05-19 23:58:02 INFO: Done loading processors!


# New DB

In [3]:
import pickle
with open('df_embed.pkl', 'rb') as f:
    df=pickle.load(f)

# Process sample

In [4]:
def process_sample(df_database: pd.DataFrame, text:str) -> pd.DataFrame:
   
    vector = vectorizer.process(text)
    print(vector.shape)

    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        return a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b))
    df_new=df_database.copy()
    df_new['similarity'] = df_new['average_embedding'].apply(lambda e: cosine_similarity(e, vector))
    top5 = df_new.nlargest(5, 'similarity').copy()
    top5['distance'] = 1 - top5['similarity']
    top5 = top5.drop(columns=['similarity'])

    return top5

In [5]:
text = "Я так понимаю рутуб до сих пор не популярен? Судя по просмотрам"

In [6]:
top5=process_sample(df, text=text)

(1024,)


In [7]:
top5

,category,question,answer,normalized_text,ner_text,average_embedding,distance
7,FAQ: Общие вопросы,Я так понимаю рутуб до сих пор не популярен? С...,Количество просмотров определенных видео завис...,я так понимать рутуб до сей пора не популярить...,Я так понимаю рутуб до сих пор не популярен? С...,"[1.59501051902771, -0.4946204125881195, 0.1764...",5.533341e-08
689,FAQ: Просмотр видео,Возможен ли просмотр фильмов на RUTUBE в VR?,На данный момент такой возможности нет. Мы учл...,возможный ли просмотр фильм на rutube в vо?,Возможен ли просмотр фильмов на ORG-A в VR?,"[1.4942021369934082, -0.05012470235427221, -1....",5.195719e-01
237,FAQ: Загрузка видео,Почему мало просмотров? Как увеличить просмотры?,"Чтобы увеличить просмотры, используйте кликбей...",почему мало просмотрыть как увеличить просмотрый,Почему мало просмотров? Как увеличить просмотры?,"[1.064292351404826, -0.13534179826577505, -0.5...",5.223098e-01
1093,FAQ: Внутренний сервис донатов RUTUBE,Можно ли посмотреть статистику по донатам по к...,"Нет, на данный момент функционал недоступен.",можно ли смотреть статистика по донатый по кон...,Можно ли посмотреть статистику по донатам по к...,"[1.1641524235407512, 0.10204730182886124, -0.4...",5.266263e-01
1333,FAQ: Регистрация блогеров 10 тыс+,"Не заблокирует ли меня, если я смотрю контент ...","Нет, ваш канал не заблокируют.",не заблокировать ли менять если я смотреть кон...,"Не заблокирует ли меня, если я смотрю контент ...","[0.7452823718388876, -0.4238361616929372, -0.2...",5.269106e-01
